# Entendendo os dados

Este notebook tem como objetivo entender de onde os dados estão vindo, para onde estão indo e qual é o estado atual da cobertura disponível no workspace.

A ideia é passar por todas as pastas de `data/`, olhando o papel de cada camada no projeto Prioriza-Leitos:

- `data/raw`: arquivos brutos baixados das fontes oficiais;
- `data/interim`: arquivos intermediários convertidos para formatos mais fáceis de ler;
- `data/processed`: tabelas analíticas prontas para features, score e visualização;
- `data/external`: dados externos auxiliares, quando existirem.

Este notebook é exploratório. Ele ajuda a entender estrutura, granularidade, qualidade inicial e cobertura dos dados, mas não substitui as validações formais do pipeline.


## Setup

Primeiro carregamos bibliotecas e criamos funções pequenas para listar arquivos, ler amostras e resumir DataFrames.

In [15]:
from pathlib import Path
import json
import zipfile

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"notebooks", "nootbooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"

DATA_DIR

PosixPath('/home/berr/GitHub/Prioriza-Leitos/data')

In [16]:
def human_size(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.1f} {unit}"
        size /= 1024


def list_files(folder: Path, recursive: bool = False) -> pd.DataFrame:
    pattern = "**/*" if recursive else "*"
    rows = []
    for path in sorted(folder.glob(pattern)):
        if path.is_file() and path.name != ".gitkeep":
            rows.append(
                {
                    "arquivo": str(path.relative_to(PROJECT_ROOT)),
                    "pasta": str(path.parent.relative_to(PROJECT_ROOT)),
                    "extensao": "".join(path.suffixes),
                    "tamanho": human_size(path.stat().st_size),
                    "bytes": path.stat().st_size,
                }
            )
    return pd.DataFrame(rows)


def read_any(path: Path) -> pd.DataFrame:
    suffixes = "".join(path.suffixes).lower()
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".csv" or suffixes.endswith(".csv.gz"):
        return pd.read_csv(path, low_memory=False)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xls", ".xlsx"}:
        return pd.read_excel(path)
    raise ValueError(f"Leitura tabular não configurada para {path.name}")


def first_data_file(folder: Path, suffixes: tuple[str, ...] = (".parquet", ".csv", ".json", ".xlsx", ".xls")) -> Path | None:
    for suffix in suffixes:
        for path in sorted(folder.glob("*")):
            if path.is_file() and path.name != ".gitkeep" and path.suffix.lower() == suffix:
                return path
    return None


def inspect_dataframe(df: pd.DataFrame, sample_rows: int = 5) -> None:
    print(f"linhas: {df.shape[0]:,}")
    print(f"colunas: {df.shape[1]:,}")
    display(df.head(sample_rows))
    display(
        pd.DataFrame(
            {
                "coluna": df.columns,
                "dtype": [str(dtype) for dtype in df.dtypes],
                "nulos": df.isna().sum().to_numpy(),
                "nulos_pct": (df.isna().mean() * 100).round(2).to_numpy(),
            }
        )
    )

## Estado atual do workspace

Atualização registrada em `2026-06-06`, após os downloads históricos executados no projeto, a inclusão da estrutura etária do Censo 2022 e a criação do de-para técnico de leitos CNES.

Cobertura bruta e tabelas auxiliares disponíveis agora:

| Fonte | Pasta | Cobertura local | Estado |
|---|---|---:|---|
| CNES LT | `data/raw/cnes/` | 84 arquivos mensais, `2019-2025` | completo no raw |
| De-para CNES leitos | `data/external/cnes_codleito_tipo_leito.csv` | 80 combinações `TP_LEITO + CODLEITO` | completo para códigos observados localmente |
| SIH RD | `data/raw/sih/` | 84 arquivos mensais, `2019-2025` | completo no raw |
| IBGE população municipal | `data/raw/ibge/` | 7 arquivos anuais, `2019-2025` | completo no raw |
| IBGE idade/sexo | `data/raw/ibge/` | Censo 2022, faixas etárias quinquenais por sexo | completo para denominadores atuais |
| SINASC DN | `data/raw/sinasc/` | 6 arquivos anuais, `2019-2024` | completo no raw para arquivos finais disponíveis |
| SIM DO | `data/raw/sim/` | 6 arquivos anuais, `2019-2024` | completo no raw para arquivos finais disponíveis |

Observações importantes:

- `CNES_RS_2024_12_2026-06-06.dbc`, `SIH_RD_RS_2024_12_2026-06-06.dbc`, `SINASC_DN_RS_2024_2026-06-06.dbc` e `SIM_DO_RS_2024_2026-06-06.dbc` já existiam e foram reaproveitados pelos scripts.
- A classificação de leitos CNES usa `TP_LEITO + CODLEITO`, não apenas `CODLEITO`, porque o código do leito pode repetir entre grupos.
- O de-para cobre leitos gerais do MVP e UTIs Adulto, Pediátrica, Neonatal, Queimado e Coronariana, além de UCIs e categorias fora do MVP.
- A população IBGE via SIDRA tabela `6579` retornou somente cabeçalho para `2022` e `2023`.
- Para `2022`, foi usado o Censo 2022 via SIDRA tabela `4714`, mantendo o mesmo formato JSON esperado pelo pipeline.
- Para `2023`, foi baixado o XLS oficial IBGE/TCU de municípios e normalizado para `IBGE_POP_RS_2023_2026-06-06.json`.
- Para idade/sexo, foi baixada a tabela SIDRA `9514` com homens/mulheres e faixas quinquenais suficientes para `pop_0_14`, `pop_15_49_feminina` e `pop_60_mais`.
- `fact_populacao_ibge.parquet` foi reconstruída com 3.479 linhas (`497 municípios x 7 anos`) e as três faixas etárias preenchidas por aproximação proporcional com base no Censo 2022.
- `data/interim/` e as facts hospitalares/materno-infantis/mortalidade ainda refletem principalmente a execução anterior. O próximo passo operacional é converter os `.dbc` históricos e reconstruir as facts para o período completo disponível.


In [ ]:
from pathlib import Path
import re

raw_checks = []
for fonte, pasta, padrao in [
    ("CNES LT", RAW_DIR / "cnes", "CNES_RS_*.dbc"),
    ("SIH RD", RAW_DIR / "sih", "SIH_RD_RS_*.dbc"),
    ("IBGE população", RAW_DIR / "ibge", "IBGE_POP_RS_*.json"),
]:
    arquivos = sorted(pasta.glob(padrao))
    anos = sorted({
        match.group(1)
        for arquivo in arquivos
        for match in [re.search(r"_(20\d{2})(?:_|$)", arquivo.name)]
        if match
    })
    raw_checks.append(
        {
            "fonte": fonte,
            "arquivos": len(arquivos),
            "anos": ", ".join(anos),
            "primeiro_arquivo": arquivos[0].name if arquivos else None,
            "ultimo_arquivo": arquivos[-1].name if arquivos else None,
        }
    )

pd.DataFrame(raw_checks)


## Data

A pasta `data/` guarda todo o fluxo de dados do projeto, desde o arquivo bruto baixado até as tabelas processadas usadas em features e score.

A regra mental aqui é:

`raw -> interim -> processed -> features/modelos/relatórios`

In [17]:
list_files(DATA_DIR, recursive=True).sort_values(["pasta", "arquivo"])

,arquivo,pasta,extensao,tamanho,bytes
0,data/interim/cnes/CNES_RS_2024_12_2026-06-06.parquet,data/interim/cnes,.parquet,41.8 KB,42791
1,data/interim/sih/SIH_RD_RS_2024_12_2026-06-06.parquet,data/interim/sih,.parquet,3.2 MB,3328076
2,data/interim/sim/SIM_DO_RS_2024_2026-06-06.parquet,data/interim/sim,.parquet,5.2 MB,5451040
3,data/interim/sinasc/SINASC_DN_RS_2024_2026-06-06.parquet,data/interim/sinasc,.parquet,3.6 MB,3808542
4,data/processed/dim_municipio_rs.parquet,data/processed,.parquet,37.0 KB,37854
5,data/processed/fact_internacoes_sih.parquet,data/processed,.parquet,573.6 KB,587371
6,data/processed/fact_leitos_cnes.parquet,data/processed,.parquet,11.7 KB,11938
7,data/processed/fact_mortalidade_sim.parquet,data/processed,.parquet,12.7 KB,13032
8,data/processed/fact_nascimentos_sinasc.parquet,data/processed,.parquet,13.1 KB,13431
9,data/processed/fact_populacao_ibge.parquet,data/processed,.parquet,9.6 KB,9797


In [18]:
summary = list_files(DATA_DIR, recursive=True)

if summary.empty:
    print("Nenhum arquivo de dados encontrado.")
else:
    (
        summary.assign(camada=summary["pasta"].str.extract(r"data/([^/]+)"))
        .groupby("camada", dropna=False)
        .agg(arquivos=("arquivo", "count"), tamanho_total_bytes=("bytes", "sum"))
        .assign(tamanho_total=lambda df: df["tamanho_total_bytes"].map(human_size))
        .reset_index()
    )

### External

**O que é?**

Camada reservada para dados externos auxiliares que não fazem parte diretamente dos downloads principais do DATASUS, IBGE ou SES/RS.

**De onde vem?**

Pode receber arquivos complementares de fontes públicas, estudos, tabelas auxiliares ou mapeamentos manuais documentados.

Links oficiais usados até agora:

- CNES/DATASUS - tipos de leito: https://cnes2.datasus.gov.br/Mod_Ind_Tipo_Leito.asp?VEstado=00
- CONASS - tabela de domínio CNES leito: https://wiki.conass.org.br/index.php?title=Tabela_de_dom%C3%ADnio_CNES_leito

**Para onde vai?**

Quando usada, essa camada deve alimentar builders específicos em `src/`, gerando tabelas em `data/processed/` ou enriquecendo features. No estado atual, contém o de-para técnico `data/external/cnes_codleito_tipo_leito.csv`.


In [19]:
list_files(EXTERNAL_DIR)

""


### Raw

**O que é?**

Camada dos dados brutos, o mais próximo possível do que foi baixado das fontes oficiais.

**De onde vem?**

Vem principalmente dos scripts em `scripts/download_*.py` e dos módulos em `src/data_acquisition/`.

Links oficiais consolidados das fontes brutas:

- CNES LT: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/LT/
- SIH RD: ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/
- SINASC DNRES: ftp://ftp.datasus.gov.br/dissemin/publicos/SINASC/1996_/Dados/DNRES/
- SIM DORES: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/
- IBGE Localidades: https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios?orderBy=nome
- IBGE SIDRA: https://apisidra.ibge.gov.br/home/ajuda
- IBGE Malha Municipal RS 2022: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/RS/
- SES/RS Regionalização: https://saude.rs.gov.br/ageplan-regionalizacao

**Estado atual**

Depois dos downloads de `2026-06-06`, o `raw` contém o histórico completo necessário para CNES LT, SIH RD e população total IBGE no recorte `2019-2025`. Também contém SINASC DN e SIM DO completos para `2019-2024`, que é o recorte inicial de arquivos finais disponíveis documentado no projeto.

**Para onde vai?**

Arquivos `.dbc` e demais formatos brutos são convertidos ou tratados para `data/interim/` e depois consolidados em `data/processed/`.


In [20]:
list_files(RAW_DIR, recursive=True).sort_values(["pasta", "arquivo"])

,arquivo,pasta,extensao,tamanho,bytes
0,data/raw/cnes/CNES_RS_2024_12_2026-06-06.dbc,data/raw/cnes,.dbc,33.6 KB,34424
1,data/raw/ibge/IBGE_POP_BR_2025_2026-06-06.xls,data/raw/ibge,.xls,826.5 KB,846336
2,data/raw/ibge/IBGE_POP_RS_2025_2026-06-06.json,data/raw/ibge,.json,90.4 KB,92587
3,data/raw/ibge/IBGE_SHAPES_RS_2022_2026-06-06.zip,data/raw/ibge,.zip,7.4 MB,7795585
4,data/raw/ses/SES_RS_REGIONALIZACAO_2025.xlsx,data/raw/ses,.xlsx,69.5 KB,71160
5,data/raw/sih/SIH_RD_RS_2024_12_2026-06-06.dbc,data/raw/sih,.dbc,5.0 MB,5257046
6,data/raw/sim/SIM_DO_RS_2024_2026-06-06.dbc,data/raw/sim,.dbc,8.0 MB,8440285
7,data/raw/sinasc/SINASC_DN_RS_2024_2026-06-06.dbc,data/raw/sinasc,.dbc,5.4 MB,5619697


#### raw/cnes

**O que é?**

Arquivos brutos de leitos do CNES/DATASUS. No projeto, eles medem oferta de leitos SUS por município, competência mensal e tipo de leito.

**De onde vem?**

Vêm do subconjunto `LT` do CNES no DATASUS.

Links oficiais:

- FTP CNES LT: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/LT/
- Página CNES Recursos Físicos/DATASUS: https://datasus.saude.gov.br/cnes-recursos-fisicos/
- CNES/DATASUS - tipos de leito: https://cnes2.datasus.gov.br/Mod_Ind_Tipo_Leito.asp?VEstado=00
- CONASS - tabela de domínio CNES leito: https://wiki.conass.org.br/index.php?title=Tabela_de_dom%C3%ADnio_CNES_leito

**Estado atual**

Há 84 arquivos `.dbc` no workspace: `2019-2025`, meses `01-12`. Essa cobertura é suficiente para construir a série histórica de oferta do MVP, desde que a conversão para `data/interim/cnes/` seja executada para todos os meses.

A classificação técnica de tipo de leito está em `data/external/cnes_codleito_tipo_leito.csv`. Ela usa a chave composta `TP_LEITO + CODLEITO` e inclui leitos gerais, UTIs, UCIs e categorias fora do MVP.

**Para onde vai?**

São convertidos para `data/interim/cnes/` e depois agregados em `fact_leitos_cnes.parquet`. A métrica principal do MVP continua sendo `QT_SUS`.


In [21]:
list_files(RAW_DIR / "cnes")

,arquivo,pasta,extensao,tamanho,bytes
0,data/raw/cnes/CNES_RS_2024_12_2026-06-06.dbc,data/raw/cnes,.dbc,33.6 KB,34424


#### raw/sih

**O que é?**

Arquivos brutos de AIH reduzida do SIH/SUS. Eles representam internações hospitalares financiadas pelo SUS.

**De onde vem?**

Vêm do subconjunto `RD` do SIH/SUS no DATASUS.

Links oficiais:

- FTP SIH RD: ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/
- Página Produção Hospitalar SIH/SUS: https://datasus.saude.gov.br/acesso-a-informacao/producao-hospitalar-sih-sus/

**Estado atual**

Há 84 arquivos `.dbc` no workspace: `2019-2025`, meses `01-12`. Essa cobertura permite calcular internações, dias de permanência, óbitos hospitalares, uso de UTI e fluxos residência-internação para o período completo do MVP após conversão.

**Para onde vai?**

São convertidos para `data/interim/sih/` e depois viram `fact_internacoes_sih.parquet`, preservando município de residência e município de internação. Essa distinção é central para não confundir demanda local com atendimento regional.


In [22]:
list_files(RAW_DIR / "sih")

,arquivo,pasta,extensao,tamanho,bytes
0,data/raw/sih/SIH_RD_RS_2024_12_2026-06-06.dbc,data/raw/sih,.dbc,5.0 MB,5257046


#### raw/sinasc

**O que é?**

Arquivos brutos do SINASC, usados para nascidos vivos, prematuridade, baixo peso e tipo de parto.

**De onde vem?**

Vêm do subconjunto `DNRES` do SINASC/DATASUS.

Links oficiais:

- FTP SINASC DNRES: ftp://ftp.datasus.gov.br/dissemin/publicos/SINASC/1996_/Dados/DNRES/
- Página SINASC/DATASUS: https://datasus.saude.gov.br/nascidos-vivos-desde-1994

**Estado atual**

Há 6 arquivos `.dbc` no workspace: `2019-2024`. Esse recorte cobre os arquivos finais inicialmente previstos para indicadores de obstetrícia e expansão neonatal.

**Para onde vai?**

São convertidos para `data/interim/sinasc/` e consolidados em `fact_nascimentos_sinasc.parquet`.


In [23]:
list_files(RAW_DIR / "sinasc")

,arquivo,pasta,extensao,tamanho,bytes
0,data/raw/sinasc/SINASC_DN_RS_2024_2026-06-06.dbc,data/raw/sinasc,.dbc,5.4 MB,5619697


#### raw/sim

**O que é?**

Arquivos brutos do SIM, usados para mortalidade total, infantil, neonatal e causas específicas.

**De onde vem?**

Vêm do subconjunto `DORES` do SIM/DATASUS.

Links oficiais:

- FTP SIM DORES: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/
- Página SIM/DATASUS: https://datasus.saude.gov.br/mortalidade-desde-1996-pela-cid-10

**Estado atual**

Há 6 arquivos `.dbc` no workspace: `2019-2024`. Esse recorte cobre os arquivos finais inicialmente previstos para indicadores de risco epidemiológico.

**Para onde vai?**

São convertidos para `data/interim/sim/` e consolidados em `fact_mortalidade_sim.parquet`.


In [24]:
list_files(RAW_DIR / "sim")

,arquivo,pasta,extensao,tamanho,bytes
0,data/raw/sim/SIM_DO_RS_2024_2026-06-06.dbc,data/raw/sim,.dbc,8.0 MB,8440285


#### raw/ibge

**O que é?**

Arquivos do IBGE: população municipal, população por idade/sexo, malhas territoriais e possíveis planilhas auxiliares.

**De onde vem?**

Vêm da API de localidades/SIDRA do IBGE ou de downloads oficiais de estimativas, Censo, TCU e malhas.

Links oficiais:

- API Localidades IBGE - municípios do RS: https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios?orderBy=nome
- SIDRA tabela 6579 - população residente estimada: https://apisidra.ibge.gov.br/values/t/6579/n6/in%20n3%2043/v/9324/p/2019-2025?formato=json
- SIDRA tabela 4714 - Censo 2022/população residente: https://apisidra.ibge.gov.br/values/t/4714/n6/in%20n3%2043/v/93/p/2022?formato=json
- SIDRA tabela 9514 - idade e sexo/Censo 2022: https://apisidra.ibge.gov.br/values/t/9514/n6/in%20n3%2043/v/93/p/2022/c2/4,5/c286/113635/c287/93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93095,93096,93097,93098,49108,49109,60040,60041,6653?formato=json
- IBGE/TCU - população dos municípios para publicação no DOU: https://www.ibge.gov.br/estatisticas/sociais/populacao/37734-relacao-da-populacao-dos-municipios-para-publicacao-no-tcu.html
- XLS IBGE/TCU 2023 municípios: https://ftp.ibge.gov.br/Informacoes_Gerais_e_Referencia/Relacao_da_Populacao_dos_Municipios_para_publicacao_no_DOU_em_2023/POP_TCU_2023_Municipios_POP2022_Malha2023.xls
- XLS IBGE POP2025: https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2025/POP2025_20251031.xls
- Malha Municipal RS 2022: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/RS/RS_Municipios_2022.zip

**Estado atual**

A população total municipal está disponível para `2019-2025`, um JSON por ano, com 497 municípios por ano.

Detalhe metodológico da população total:

- `2019`, `2020`, `2021`, `2024` e `2025` vieram da SIDRA tabela `6579`;
- `2022` veio do Censo 2022, SIDRA tabela `4714`;
- `2023` veio do XLS oficial IBGE/TCU de municípios, normalizado para o mesmo formato JSON usado pelo pipeline.

A população por idade/sexo veio da SIDRA tabela `9514`, Censo 2022. O recorte usa homens/mulheres e faixas etárias quinquenais suficientes para preencher os denominadores atuais.

**Para onde vai?**

Alimenta `dim_municipio_rs.parquet`, `fact_populacao_ibge.parquet` e futuras análises espaciais. As faixas `pop_0_14`, `pop_15_49_feminina` e `pop_60_mais` são preenchidas aplicando a estrutura municipal do Censo 2022 como proporção sobre a população total anual. Essa é uma aproximação documentada, não uma estimativa oficial por idade para cada ano.


In [25]:
list_files(RAW_DIR / "ibge")

,arquivo,pasta,extensao,tamanho,bytes
0,data/raw/ibge/IBGE_POP_BR_2025_2026-06-06.xls,data/raw/ibge,.xls,826.5 KB,846336
1,data/raw/ibge/IBGE_POP_RS_2025_2026-06-06.json,data/raw/ibge,.json,90.4 KB,92587
2,data/raw/ibge/IBGE_SHAPES_RS_2022_2026-06-06.zip,data/raw/ibge,.zip,7.4 MB,7795585


In [26]:
arquivo = first_data_file(RAW_DIR / "ibge")
print(arquivo)

if arquivo is not None:
    df = read_any(arquivo)
    inspect_dataframe(df)

/home/berr/GitHub/Prioriza-Leitos/data/raw/ibge/IBGE_POP_RS_2025_2026-06-06.json
linhas: 498
colunas: 11


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
1,6,Município,45,Pessoas,4251,4300034,Aceguá - RS,9324,População residente estimada,2025,2025
2,6,Município,45,Pessoas,4003,4300059,Água Santa - RS,9324,População residente estimada,2025,2025
3,6,Município,45,Pessoas,16341,4300109,Agudo - RS,9324,População residente estimada,2025,2025
4,6,Município,45,Pessoas,6843,4300208,Ajuricaba - RS,9324,População residente estimada,2025,2025


,coluna,dtype,nulos,nulos_pct
0,NC,object,0,0.0
1,NN,object,0,0.0
2,MC,object,0,0.0
3,MN,object,0,0.0
4,V,object,0,0.0
5,D1C,object,0,0.0
6,D1N,object,0,0.0
7,D2C,object,0,0.0
8,D2N,object,0,0.0
9,D3C,object,0,0.0


#### raw/ses

**O que é?**

Planilhas da Secretaria Estadual de Saúde do RS, usadas para regionalização, regiões de saúde e macrorregiões.

**De onde vem?**

Vem de fonte pública da SES/RS, baixada durante a construção da dimensão municipal quando necessário.

Links oficiais:

- Página SES/RS Regionalização: https://saude.rs.gov.br/ageplan-regionalizacao
- XLS Regionalização em Saúde 2025: https://saude.rs.gov.br/upload/arquivos/202507/31162659-regionalizacao-em-saude-2025.xlsx

**Para onde vai?**

Enriquece `dim_municipio_rs.parquet` com região e macrorregião de saúde.


In [27]:
list_files(RAW_DIR / "ses")

,arquivo,pasta,extensao,tamanho,bytes
0,data/raw/ses/SES_RS_REGIONALIZACAO_2025.xlsx,data/raw/ses,.xlsx,69.5 KB,71160


In [28]:
arquivo = first_data_file(RAW_DIR / "ses")
print(arquivo)

if arquivo is not None:
    df = read_any(arquivo)
    inspect_dataframe(df)

/home/berr/GitHub/Prioriza-Leitos/data/raw/ses/SES_RS_REGIONALIZACAO_2025.xlsx
linhas: 497
colunas: 8


,CódigoIBGE,Município,CRS,Região de Saúde,Macrorregião de Saúde,COREDES (SPGG 03/08/2022),RegiãoFuncional,ASSOCIAÇÃO FAMURS
0,430060,Alvorada,1,10 - Capital e Vale do Gravataí,Metropolitana,Metropolitano Delta do Jacuí,RF1,GRANPAL
1,430085,Arambaré,1,09 - Carbonífera/Costa Doce,Metropolitana,Centro-Sul,RF1,ACOSTADOCE
2,430087,Araricá,1,07 - Vale dos Sinos,Metropolitana,Vale do Rio dos Sinos,RF1,AMVARS
3,430110,Arroio dos Ratos,1,09 - Carbonífera/Costa Doce,Metropolitana,Centro-Sul,RF1,ASMURC
4,430165,Barão,1,08 - Vale do Caí e Metropolitana,Metropolitana,Vale do Caí,RF1,AMVARC


,coluna,dtype,nulos,nulos_pct
0,CódigoIBGE,int64,0,0.0
1,Município,object,0,0.0
2,CRS,int64,0,0.0
3,Região de Saúde,object,0,0.0
4,Macrorregião de Saúde,object,0,0.0
5,COREDES (SPGG 03/08/2022),object,0,0.0
6,RegiãoFuncional,object,0,0.0
7,ASSOCIAÇÃO FAMURS,object,0,0.0


#### raw/datasus

**O que é?**

Pasta auxiliar para downloads ou artefatos gerais do DATASUS.

**De onde vem?**

Pode receber dados brutos baixados por rotinas mais genéricas de aquisição.

Links oficiais gerais:

- DATASUS - transferência de arquivos: https://datasus.saude.gov.br/transferencia-de-arquivos/
- FTP público DATASUS: ftp://ftp.datasus.gov.br/dissemin/publicos/

**Para onde vai?**

Se houver arquivos aqui, eles devem ser encaminhados para conversão em `data/interim/` ou reorganizados nas pastas específicas de fonte.


In [29]:
list_files(RAW_DIR / "datasus")

""


### Interim

**O que é?**

Camada intermediária. Aqui ficam arquivos já convertidos para formato tabular legível pelo pandas, principalmente Parquet.

**De onde vem?**

Vem da conversão dos dados brutos em `data/raw/`, especialmente arquivos `.dbc` do DATASUS.

**Para onde vai?**

Vai para os builders em `src/data_cleaning/`, que criam as facts em `data/processed/`.

In [30]:
list_files(INTERIM_DIR, recursive=True).sort_values(["pasta", "arquivo"])

,arquivo,pasta,extensao,tamanho,bytes
0,data/interim/cnes/CNES_RS_2024_12_2026-06-06.parquet,data/interim/cnes,.parquet,41.8 KB,42791
1,data/interim/sih/SIH_RD_RS_2024_12_2026-06-06.parquet,data/interim/sih,.parquet,3.2 MB,3328076
2,data/interim/sim/SIM_DO_RS_2024_2026-06-06.parquet,data/interim/sim,.parquet,5.2 MB,5451040
3,data/interim/sinasc/SINASC_DN_RS_2024_2026-06-06.parquet,data/interim/sinasc,.parquet,3.6 MB,3808542


#### interim/cnes

**O que é?**

Dados de leitos CNES já convertidos para Parquet.

**De onde vem?**

Vem de `data/raw/cnes/*.dbc` após a conversão.

**Para onde vai?**

Vai para `fact_leitos_cnes.parquet`, agregando leitos SUS por município, ano, mês e tipo de leito.

In [31]:
list_files(INTERIM_DIR / "cnes")

,arquivo,pasta,extensao,tamanho,bytes
0,data/interim/cnes/CNES_RS_2024_12_2026-06-06.parquet,data/interim/cnes,.parquet,41.8 KB,42791


In [32]:
arquivo = first_data_file(INTERIM_DIR / "cnes", suffixes=(".parquet", ".csv"))
print(arquivo)

if arquivo is not None:
    df = read_any(arquivo)
    inspect_dataframe(df)
    display(df.head())

/home/berr/GitHub/Prioriza-Leitos/data/interim/cnes/CNES_RS_2024_12_2026-06-06.parquet
linhas: 2,994
colunas: 28


,CNES,CODUFMUN,REGSAUDE,MICR_REG,DISTRSAN,DISTRADM,TPGESTAO,PF_PJ,CPF_CNPJ,NIV_DEP,CNPJ_MAN,ESFERA_A,ATIVIDAD,RETENCAO,NATUREZA,CLIENTEL,TP_UNID,TURNO_AT,NIV_HIER,TERCEIRO,TP_LEITO,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN,NAT_JUR
0,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,2,33,17,0,15,2,202412,3999
1,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,4,10,3,0,2,1,202412,3999
2,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,1,03,11,0,7,4,202412,3999
3,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,6,34,1,0,1,0,202412,3999
4,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,4,43,4,0,3,1,202412,3999


,coluna,dtype,nulos,nulos_pct
0,CNES,object,0,0.0
1,CODUFMUN,object,0,0.0
2,REGSAUDE,object,0,0.0
3,MICR_REG,object,0,0.0
4,DISTRSAN,object,0,0.0
5,DISTRADM,object,0,0.0
6,TPGESTAO,object,0,0.0
7,PF_PJ,object,0,0.0
8,CPF_CNPJ,object,0,0.0
9,NIV_DEP,object,0,0.0


,CNES,CODUFMUN,REGSAUDE,MICR_REG,DISTRSAN,DISTRADM,TPGESTAO,PF_PJ,CPF_CNPJ,NIV_DEP,CNPJ_MAN,ESFERA_A,ATIVIDAD,RETENCAO,NATUREZA,CLIENTEL,TP_UNID,TURNO_AT,NIV_HIER,TERCEIRO,TP_LEITO,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN,NAT_JUR
0,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,2,33,17,0,15,2,202412,3999
1,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,4,10,3,0,2,1,202412,3999
2,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,1,03,11,0,7,4,202412,3999
3,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,6,34,1,0,1,0,202412,3999
4,2262010,430003,7,,,,E,3,91569038000135,1,00000000000000,,04,,,03,05,06,,,4,43,4,0,3,1,202412,3999


#### interim/sih

**O que é?**

Internações hospitalares do SIH/SUS já convertidas para Parquet.

**De onde vem?**

Vem de `data/raw/sih/*.dbc` após conversão.

**Para onde vai?**

Vai para `fact_internacoes_sih.parquet`, mantendo residência, internação, diagnóstico, procedimento, dias, valor, óbito e uso de UTI.

In [33]:
list_files(INTERIM_DIR / "sih")

,arquivo,pasta,extensao,tamanho,bytes
0,data/interim/sih/SIH_RD_RS_2024_12_2026-06-06.parquet,data/interim/sih,.parquet,3.2 MB,3328076


In [34]:
arquivo = first_data_file(INTERIM_DIR / "sih", suffixes=(".parquet", ".csv"))
print(arquivo)

if arquivo is not None:
    df = read_any(arquivo)
    inspect_dataframe(df)
    display(df.head())

/home/berr/GitHub/Prioriza-Leitos/data/interim/sih/SIH_RD_RS_2024_12_2026-06-06.parquet
linhas: 67,246
colunas: 113


,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,VAL_SANGUE,VAL_SADTSR,VAL_TRANSP,VAL_OBSANG,VAL_PED1AC,VAL_TOT,VAL_UTI,US_TOT,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,COBRANCA,NATUREZA,NAT_JUR,GESTAO,RUBRICA,IND_VDRL,MUNIC_MOV,COD_IDADE,IDADE,DIAS_PERM,MORTE,NACIONAL,NUM_PROC,CAR_INT,TOT_PT_SP,CPF_AUT,HOMONIMO,NUM_FILHOS,INSTRU,CID_NOTIF,CONTRACEP1,CONTRACEP2,GESTRISCO,INSC_PN,SEQ_AIH5,CBOR,CNAER,VINCPREV,GESTOR_COD,GESTOR_TP,GESTOR_CPF,GESTOR_DT,CNES,CNPJ_MANT,INFEHOSP,CID_ASSO,CID_MORTE,COMPLEX,FINANC,FAEC_TP,REGCT,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9
0,430000,2024,12,01,92021062000106,4324109335040,1,99010000,431410,19910630,3,0,0,0,0,00,0,0,0,0,0,3,0411010034,0411010034,1050.16,276.43,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1326.59,0.0,227.93,20241223,20241226,O759,0000,61,00,3999,2,0,0,431410,4,33,3,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00014,1,000059470925068,,2246988,,,0000,0000,02,06,,0000,02,0000,15201,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0
1,430000,2024,12,01,92021062000106,4324109487929,1,99670000,431610,20230821,1,0,0,0,0,00,0,0,0,0,7,7,0303010061,0415040035,869.46,234.78,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1104.24,0.0,189.73,20241110,20241117,R02,0000,12,00,3999,2,0,0,431410,4,1,7,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,,2246988,,,0000,0000,02,06,,0000,05,0077,15310,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0
2,430000,2024,12,01,92021062000106,4324109487930,1,99670959,431610,20210926,1,0,0,0,0,00,0,0,0,0,7,9,0407020039,0407020047,458.32,205.04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,663.36,0.0,113.97,20241121,20241130,K359,0000,12,00,3999,2,0,0,431410,4,3,9,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,,2246988,,,0000,0000,02,06,,0000,01,0000,15311,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0
3,430000,2024,12,01,92021062000106,4324109487940,1,99670000,431610,19671121,3,0,0,0,5,75,0,0,0,0,0,10,0406010137,0406010137,12240.89,12992.68,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25233.57,3000.0,4335.66,20241104,20241118,I718,0000,27,00,3999,2,0,0,431410,4,56,14,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,,2246988,,,0000,0000,03,04,040082,0000,01,0000,15312,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0
4,430000,2024,12,01,92021062000106,4324109487951,1,99670000,431610,19570725,1,0,0,0,0,00,0,0,0,0,0,7,0407040129,0407030255,1715.10,548.96,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2264.06,0.0,389.01,20241122,20241128,K805,0000,27,00,3999,2,0,0,431410,4,67,6,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00014,1,000059470925068,,2246988,,,0000,0000,03,04,040024,0000,01,0000,15313,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0


,coluna,dtype,nulos,nulos_pct
0,UF_ZI,object,0,0.0
1,ANO_CMPT,object,0,0.0
2,MES_CMPT,object,0,0.0
3,ESPEC,object,0,0.0
4,CGC_HOSP,object,0,0.0
...,...,...,...,...
108,TPDISEC5,object,0,0.0
109,TPDISEC6,object,0,0.0
110,TPDISEC7,object,0,0.0
111,TPDISEC8,object,0,0.0


,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,VAL_SANGUE,VAL_SADTSR,VAL_TRANSP,VAL_OBSANG,VAL_PED1AC,VAL_TOT,VAL_UTI,US_TOT,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,COBRANCA,NATUREZA,NAT_JUR,GESTAO,RUBRICA,IND_VDRL,MUNIC_MOV,COD_IDADE,IDADE,DIAS_PERM,MORTE,NACIONAL,NUM_PROC,CAR_INT,TOT_PT_SP,CPF_AUT,HOMONIMO,NUM_FILHOS,INSTRU,CID_NOTIF,CONTRACEP1,CONTRACEP2,GESTRISCO,INSC_PN,SEQ_AIH5,CBOR,CNAER,VINCPREV,GESTOR_COD,GESTOR_TP,GESTOR_CPF,GESTOR_DT,CNES,CNPJ_MANT,INFEHOSP,CID_ASSO,CID_MORTE,COMPLEX,FINANC,FAEC_TP,REGCT,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9
0,430000,2024,12,01,92021062000106,4324109335040,1,99010000,431410,19910630,3,0,0,0,0,00,0,0,0,0,0,3,0411010034,0411010034,1050.16,276.43,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1326.59,0.0,227.93,20241223,20241226,O759,0000,61,00,3999,2,0,0,431410,4,33,3,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00014,1,000059470925068,,2246988,,,0000,0000,02,06,,0000,02,0000,15201,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0
1,430000,2024,12,01,92021062000106,4324109487929,1,99670000,431610,20230821,1,0,0,0,0,00,0,0,0,0,7,7,0303010061,0415040035,869.46,234.78,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1104.24,0.0,189.73,20241110,20241117,R02,0000,12,00,3999,2,0,0,431410,4,1,7,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,,2246988,,,0000,0000,02,06,,0000,05,0077,15310,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0
2,430000,2024,12,01,92021062000106,4324109487930,1,99670959,431610,20210926,1,0,0,0,0,00,0,0,0,0,7,9,0407020039,0407020047,458.32,205.04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,663.36,0.0,113.97,20241121,20241130,K359,0000,12,00,3999,2,0,0,431410,4,3,9,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,,2246988,,,0000,0000,02,06,,0000,01,0000,15311,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0
3,430000,2024,12,01,92021062000106,4324109487940,1,99670000,431610,19671121,3,0,0,0,5,75,0,0,0,0,0,10,0406010137,0406010137,12240.89,12992.68,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25233.57,3000.0,4335.66,20241104,20241118,I718,0000,27,00,3999,2,0,0,431410,4,56,14,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,,2246988,,,0000,0000,03,04,040082,0000,01,0000,15312,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0
4,430000,2024,12,01,92021062000106,4324109487951,1,99670000,431610,19570725,1,0,0,0,0,00,0,0,0,0,0,7,0407040129,0407030255,1715.10,548.96,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2264.06,0.0,389.01,20241122,20241128,K805,0000,27,00,3999,2,0,0,431410,4,67,6,0,010,,02,0,,0,0,0,,00,00,1,000000000000,000,000000,000,0,00014,1,000059470925068,,2246988,,,0000,0000,03,04,040024,0000,01,0000,15313,HE43000001N202412.DTS,,,0.0,0.0,0.0,0.0,0.0,00,,,,,,,,,,0,0,0,0,0,0,0,0,0


#### interim/sinasc

**O que é?**

Declarações de nascidos vivos já convertidas para formato tabular.

**De onde vem?**

Vem de `data/raw/sinasc/*.dbc`.

**Para onde vai?**

Vai para `fact_nascimentos_sinasc.parquet`, agregando indicadores materno-infantis por município e ano.

In [35]:
list_files(INTERIM_DIR / "sinasc")

,arquivo,pasta,extensao,tamanho,bytes
0,data/interim/sinasc/SINASC_DN_RS_2024_2026-06-06.parquet,data/interim/sinasc,.parquet,3.6 MB,3808542


In [36]:
arquivo = first_data_file(INTERIM_DIR / "sinasc", suffixes=(".parquet", ".csv"))
print(arquivo)

if arquivo is not None:
    df = read_any(arquivo)
    inspect_dataframe(df)
    display(df.head())

/home/berr/GitHub/Prioriza-Leitos/data/interim/sinasc/SINASC_DN_RS_2024_2026-06-06.parquet
linhas: 111,988
colunas: 61


,ORIGEM,CODESTAB,CODMUNNASC,LOCNASC,IDADEMAE,ESTCIVMAE,ESCMAE,CODOCUPMAE,QTDFILVIVO,QTDFILMORT,CODMUNRES,GESTACAO,GRAVIDEZ,PARTO,CONSULTAS,DTNASC,HORANASC,SEXO,APGAR1,APGAR5,RACACOR,PESO,IDANOMAL,DTCADASTRO,CODANOMAL,NUMEROLOTE,VERSAOSIST,DTRECEBIM,DIFDATA,DTRECORIGA,NATURALMAE,CODMUNNATU,CODUFNATU,ESCMAE2010,SERIESCMAE,DTNASCMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,IDADEPAI,DTULTMENST,SEMAGESTAC,TPMETESTIM,CONSPRENAT,MESPRENAT,TPAPRESENT,STTRABPART,STCESPARTO,TPNASCASSI,TPFUNCRESP,TPDOCRESP,DTDECLARAC,ESCMAEAGR1,STDNEPIDEM,STDNNOVA,CODPAISRES,TPROBSON,PARIDADE,KOTELCHUCK,CONTADOR
0,1,2018055,130260,1,35,2,5,214205,00,00,431800,5,1,1,4,09032024,1516,2,09,10,4,3174,2,16042024,,20240020,3.2.50,23042024,045,,813,130260,13,5,,04101988,4,00,00,00,28,,39,2,10,03,1,2,3,1,1,4,09032024,08,1,1,1,01,0,5,34028
1,1,9561501,250750,1,39,2,5,223605,01,00,432240,5,1,1,4,09012024,0945,1,06,08,1,3960,2,24012024,,20240003,3.2.01,25012024,016,,825,250750,25,5,,14011984,1,01,00,01,39,13042023,38,8,15,01,1,2,3,1,2,3,10012024,08,1,1,1,05,1,5,280367
2,1,2427427,261160,1,32,5,3,999992,03,02,432180,5,1,1,3,21012024,2245,2,10,10,4,3120,2,29022024,,20240010,3.2.50,08032024,047,,826,261180,26,1,4,12011992,4,05,03,00,,,39,2,05,03,1,2,3,1,1,2,21012024,02,1,1,1,03,1,3,327678
3,1,5714397,280030,1,23,1,4,999992,01,00,430910,4,1,1,2,28032024,2240,2,07,08,4,1506,2,12042024,,20240009,3.2.02,10052024,043,,828,280030,28,3,1,23052000,4,01,00,01,,,33,1,03,01,1,2,3,1,5,4,28032024,05,1,1,1,10,1,3,381939
4,1,4026896,291460,1,23,1,4,622020,00,00,431600,5,1,2,4,28042024,0925,1,07,08,2,3550,2,30102025,,20250045,3.2.50,05112025,556,,829,292925,29,3,3,13092000,2,00,00,00,,15072023,40,8,13,03,1,1,2,1,2,3,28042024,06,1,1,1,02,0,5,422097


,coluna,dtype,nulos,nulos_pct
0,ORIGEM,object,0,0.0
1,CODESTAB,object,0,0.0
2,CODMUNNASC,object,0,0.0
3,LOCNASC,object,0,0.0
4,IDADEMAE,object,0,0.0
...,...,...,...,...
56,CODPAISRES,object,0,0.0
57,TPROBSON,object,0,0.0
58,PARIDADE,object,0,0.0
59,KOTELCHUCK,object,0,0.0


,ORIGEM,CODESTAB,CODMUNNASC,LOCNASC,IDADEMAE,ESTCIVMAE,ESCMAE,CODOCUPMAE,QTDFILVIVO,QTDFILMORT,CODMUNRES,GESTACAO,GRAVIDEZ,PARTO,CONSULTAS,DTNASC,HORANASC,SEXO,APGAR1,APGAR5,RACACOR,PESO,IDANOMAL,DTCADASTRO,CODANOMAL,NUMEROLOTE,VERSAOSIST,DTRECEBIM,DIFDATA,DTRECORIGA,NATURALMAE,CODMUNNATU,CODUFNATU,ESCMAE2010,SERIESCMAE,DTNASCMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,IDADEPAI,DTULTMENST,SEMAGESTAC,TPMETESTIM,CONSPRENAT,MESPRENAT,TPAPRESENT,STTRABPART,STCESPARTO,TPNASCASSI,TPFUNCRESP,TPDOCRESP,DTDECLARAC,ESCMAEAGR1,STDNEPIDEM,STDNNOVA,CODPAISRES,TPROBSON,PARIDADE,KOTELCHUCK,CONTADOR
0,1,2018055,130260,1,35,2,5,214205,00,00,431800,5,1,1,4,09032024,1516,2,09,10,4,3174,2,16042024,,20240020,3.2.50,23042024,045,,813,130260,13,5,,04101988,4,00,00,00,28,,39,2,10,03,1,2,3,1,1,4,09032024,08,1,1,1,01,0,5,34028
1,1,9561501,250750,1,39,2,5,223605,01,00,432240,5,1,1,4,09012024,0945,1,06,08,1,3960,2,24012024,,20240003,3.2.01,25012024,016,,825,250750,25,5,,14011984,1,01,00,01,39,13042023,38,8,15,01,1,2,3,1,2,3,10012024,08,1,1,1,05,1,5,280367
2,1,2427427,261160,1,32,5,3,999992,03,02,432180,5,1,1,3,21012024,2245,2,10,10,4,3120,2,29022024,,20240010,3.2.50,08032024,047,,826,261180,26,1,4,12011992,4,05,03,00,,,39,2,05,03,1,2,3,1,1,2,21012024,02,1,1,1,03,1,3,327678
3,1,5714397,280030,1,23,1,4,999992,01,00,430910,4,1,1,2,28032024,2240,2,07,08,4,1506,2,12042024,,20240009,3.2.02,10052024,043,,828,280030,28,3,1,23052000,4,01,00,01,,,33,1,03,01,1,2,3,1,5,4,28032024,05,1,1,1,10,1,3,381939
4,1,4026896,291460,1,23,1,4,622020,00,00,431600,5,1,2,4,28042024,0925,1,07,08,2,3550,2,30102025,,20250045,3.2.50,05112025,556,,829,292925,29,3,3,13092000,2,00,00,00,,15072023,40,8,13,03,1,1,2,1,2,3,28042024,06,1,1,1,02,0,5,422097


#### interim/sim

**O que é?**

Declarações de óbito já convertidas para formato tabular.

**De onde vem?**

Vem de `data/raw/sim/*.dbc`.

**Para onde vai?**

Vai para `fact_mortalidade_sim.parquet`, agregando óbitos por município, ano e grupos de causa.

In [37]:
list_files(INTERIM_DIR / "sim")

,arquivo,pasta,extensao,tamanho,bytes
0,data/interim/sim/SIM_DO_RS_2024_2026-06-06.parquet,data/interim/sim,.parquet,5.2 MB,5451040


In [38]:
arquivo = first_data_file(INTERIM_DIR / "sim", suffixes=(".parquet", ".csv"))
print(arquivo)

if arquivo is not None:
    df = read_any(arquivo)
    inspect_dataframe(df)
    display(df.head())

/home/berr/GitHub/Prioriza-Leitos/data/interim/sim/SIM_DO_RS_2024_2026-06-06.parquet
linhas: 101,480
colunas: 87


,ORIGEM,TIPOBITO,DTOBITO,HORAOBITO,NATURAL,CODMUNNATU,DTNASC,IDADE,SEXO,RACACOR,ESTCIV,ESC,ESC2010,SERIESCFAL,OCUP,CODMUNRES,LOCOCOR,CODESTAB,ESTABDESCR,CODMUNOCOR,IDADEMAE,ESCMAE,ESCMAE2010,SERIESCMAE,OCUPMAE,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,SEMAGESTAC,GESTACAO,PARTO,OBITOPARTO,PESO,TPMORTEOCO,OBITOGRAV,OBITOPUERP,ASSISTMED,EXAME,CIRURGIA,NECROPSIA,LINHAA,LINHAB,LINHAC,LINHAD,LINHAII,CAUSABAS,CB_PRE,COMUNSVOIM,DTATESTADO,CIRCOBITO,ACIDTRAB,FONTE,NUMEROLOTE,TPPOS,DTINVESTIG,CAUSABAS_O,DTCADASTRO,ATESTANTE,STCODIFICA,CODIFICADO,VERSAOSIST,VERSAOSCB,FONTEINV,DTRECEBIM,ATESTADO,DTRECORIGA,CAUSAMAT,ESCMAEAGR1,ESCFALAGR1,STDOEPIDEM,STDONOVA,DIFDATA,NUDIASOBCO,NUDIASOBIN,DTCADINV,TPOBITOCOR,DTCONINV,FONTES,TPRESGINFO,TPNIVELINV,NUDIASINF,DTCADINF,MORTEPARTO,DTCONCASO,FONTESINF,ALTCAUSA,CONTADOR
0,1,2,01012024,2229,843,430690,25101953,470,1,1,9,9,9,,,430690,1,2234432,,430690,,,,,,,,,,,,,,,,,,,,,*J960,*A419,*R092,*C349,*K721,C349,,,07012024,,,,20240006,,,C349,08012024,1,S,S,3.2.30,3.4,,05022024,J960/A419 /R092 /C349 /K721,08012024,,,09,0,1,035,,,,,,,,,,,,,,,18
1,1,2,01012024,1300,843,430570,16061940,483,2,1,2,3,1,4,612005,431843,1,2244306,,431690,,,,,,,,,,,,,,,,,1,,,2,*A419,*J189,,,*I694*E669,I694,,,01012024,,,,20240002,N,,I694,08012024,2,S,S,3.2.30,3.4,,10012024,A419/J189 / / /I694*E669,10012024,,,02,0,1,009,,,,,,,,,,,,,,,41
2,1,2,01012024,2215,843,431447,26071948,475,1,1,2,4,3,3,411010,430800,1,2244276,,431690,,,,,,,,,,,,,,,,,1,,,2,*C259,,,,,C259,,,01012024,,,,20240002,N,,C259,08012024,2,S,S,3.2.30,3.4,,10012024,C259/ / / /,10012024,,,06,0,1,009,,,,,,,,,,,,,,,42
3,1,2,01012024,1715,843,431410,10091957,466,1,1,1,,,,999993,431410,5,,,431410,,,,,,,,,,,,,,,,,1,,,2,*J969,*J189,*F107,,*F102*E039,F107,,431410,01012024,,,,20240032,S,25072024,J189,11012024,4,S,S,3.2.30,3.4,8,26072024,J969/J189 /F107 / /F102*E039,12012024,,,,0,1,207,,,,,,,,,,,,,,,108
4,1,2,01012024,0142,843,431690,21071977,446,2,1,1,4,2,8,999992,431690,1,9575936,,431690,,,,,,,,,,,,,,8,2,3,1,,,2,*A419,*J189,*I64X,,,I64,,,01012024,,,,20240007,S,11012024,I64,08012024,5,S,S,3.2.30,3.4,3,14022024,A419/J189 /I64X / /,10012024,,,04,0,1,044,10,,30012024,8,11012024,,,M,,,,,,,111


,coluna,dtype,nulos,nulos_pct
0,ORIGEM,object,0,0.0
1,TIPOBITO,object,0,0.0
2,DTOBITO,object,0,0.0
3,HORAOBITO,object,0,0.0
4,NATURAL,object,0,0.0
...,...,...,...,...
82,MORTEPARTO,object,0,0.0
83,DTCONCASO,object,0,0.0
84,FONTESINF,object,0,0.0
85,ALTCAUSA,object,0,0.0


,ORIGEM,TIPOBITO,DTOBITO,HORAOBITO,NATURAL,CODMUNNATU,DTNASC,IDADE,SEXO,RACACOR,ESTCIV,ESC,ESC2010,SERIESCFAL,OCUP,CODMUNRES,LOCOCOR,CODESTAB,ESTABDESCR,CODMUNOCOR,IDADEMAE,ESCMAE,ESCMAE2010,SERIESCMAE,OCUPMAE,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,SEMAGESTAC,GESTACAO,PARTO,OBITOPARTO,PESO,TPMORTEOCO,OBITOGRAV,OBITOPUERP,ASSISTMED,EXAME,CIRURGIA,NECROPSIA,LINHAA,LINHAB,LINHAC,LINHAD,LINHAII,CAUSABAS,CB_PRE,COMUNSVOIM,DTATESTADO,CIRCOBITO,ACIDTRAB,FONTE,NUMEROLOTE,TPPOS,DTINVESTIG,CAUSABAS_O,DTCADASTRO,ATESTANTE,STCODIFICA,CODIFICADO,VERSAOSIST,VERSAOSCB,FONTEINV,DTRECEBIM,ATESTADO,DTRECORIGA,CAUSAMAT,ESCMAEAGR1,ESCFALAGR1,STDOEPIDEM,STDONOVA,DIFDATA,NUDIASOBCO,NUDIASOBIN,DTCADINV,TPOBITOCOR,DTCONINV,FONTES,TPRESGINFO,TPNIVELINV,NUDIASINF,DTCADINF,MORTEPARTO,DTCONCASO,FONTESINF,ALTCAUSA,CONTADOR
0,1,2,01012024,2229,843,430690,25101953,470,1,1,9,9,9,,,430690,1,2234432,,430690,,,,,,,,,,,,,,,,,,,,,*J960,*A419,*R092,*C349,*K721,C349,,,07012024,,,,20240006,,,C349,08012024,1,S,S,3.2.30,3.4,,05022024,J960/A419 /R092 /C349 /K721,08012024,,,09,0,1,035,,,,,,,,,,,,,,,18
1,1,2,01012024,1300,843,430570,16061940,483,2,1,2,3,1,4,612005,431843,1,2244306,,431690,,,,,,,,,,,,,,,,,1,,,2,*A419,*J189,,,*I694*E669,I694,,,01012024,,,,20240002,N,,I694,08012024,2,S,S,3.2.30,3.4,,10012024,A419/J189 / / /I694*E669,10012024,,,02,0,1,009,,,,,,,,,,,,,,,41
2,1,2,01012024,2215,843,431447,26071948,475,1,1,2,4,3,3,411010,430800,1,2244276,,431690,,,,,,,,,,,,,,,,,1,,,2,*C259,,,,,C259,,,01012024,,,,20240002,N,,C259,08012024,2,S,S,3.2.30,3.4,,10012024,C259/ / / /,10012024,,,06,0,1,009,,,,,,,,,,,,,,,42
3,1,2,01012024,1715,843,431410,10091957,466,1,1,1,,,,999993,431410,5,,,431410,,,,,,,,,,,,,,,,,1,,,2,*J969,*J189,*F107,,*F102*E039,F107,,431410,01012024,,,,20240032,S,25072024,J189,11012024,4,S,S,3.2.30,3.4,8,26072024,J969/J189 /F107 / /F102*E039,12012024,,,,0,1,207,,,,,,,,,,,,,,,108
4,1,2,01012024,0142,843,431690,21071977,446,2,1,1,4,2,8,999992,431690,1,9575936,,431690,,,,,,,,,,,,,,8,2,3,1,,,2,*A419,*J189,*I64X,,,I64,,,01012024,,,,20240007,S,11012024,I64,08012024,5,S,S,3.2.30,3.4,3,14022024,A419/J189 /I64X / /,10012024,,,04,0,1,044,10,,30012024,8,11012024,,,M,,,,,,,111


### Processed

**O que é?**

Camada das tabelas analíticas do projeto. São dados mais limpos, agregados e com granularidade definida.

**De onde vem?**

Vem dos builders em `src/data_cleaning/` e de scripts como `scripts/build_dim_municipio.py`.

**Para onde vai?**

Vai para engenharia de features, modelos, scores, visualizações, relatórios e site.

In [39]:
list_files(PROCESSED_DIR).sort_values("arquivo")

,arquivo,pasta,extensao,tamanho,bytes
0,data/processed/dim_municipio_rs.parquet,data/processed,.parquet,37.0 KB,37854
1,data/processed/fact_internacoes_sih.parquet,data/processed,.parquet,573.6 KB,587371
2,data/processed/fact_leitos_cnes.parquet,data/processed,.parquet,11.7 KB,11938
3,data/processed/fact_mortalidade_sim.parquet,data/processed,.parquet,12.7 KB,13032
4,data/processed/fact_nascimentos_sinasc.parquet,data/processed,.parquet,13.1 KB,13431
5,data/processed/fact_populacao_ibge.parquet,data/processed,.parquet,9.6 KB,9797


In [40]:
processed_overview = []

for arquivo in sorted(PROCESSED_DIR.glob("*.parquet")):
    df_tmp = pd.read_parquet(arquivo)
    processed_overview.append(
        {
            "arquivo": arquivo.name,
            "linhas": len(df_tmp),
            "colunas": df_tmp.shape[1],
            "colunas_nome": ", ".join(df_tmp.columns),
        }
    )

pd.DataFrame(processed_overview)

,arquivo,linhas,colunas,colunas_nome
0,dim_municipio_rs.parquet,497,10,"municipio_id_ibge, municipio_nome, municipio_nome_normalizado, uf, regiao_saude, macroregiao_saude, latitude, longit..."
1,fact_internacoes_sih.parquet,67069,12,"municipio_residencia_id, municipio_internacao_id, ano, mes, grupo_procedimento, diagnostico_cid, idade, sexo, dias_p..."
2,fact_leitos_cnes.parquet,1007,8,"municipio_id, ano, mes, tipo_leito, leitos_existentes, leitos_sus, leitos_nao_sus, estabelecimentos_com_leito"
3,fact_mortalidade_sim.parquet,497,8,"municipio_id, ano, obitos_total, obitos_neonatais, obitos_infantis, obitos_cardiovasculares, obitos_respiratorios, o..."
4,fact_nascimentos_sinasc.parquet,497,7,"municipio_id, ano, nascidos_vivos, prematuros, baixo_peso, partos_cesareos, partos_vaginais"
5,fact_populacao_ibge.parquet,497,6,"municipio_id, ano, populacao_total, pop_0_14, pop_15_49_feminina, pop_60_mais"


#### processed/dim_municipio_rs.parquet

**O que é?**

Dimensão municipal do Rio Grande do Sul. É a tabela de referência para códigos IBGE, nomes, região de saúde, macrorregião e coordenadas.

**De onde vem?**

Vem de IBGE e SES/RS, construída por `scripts/build_dim_municipio.py`.

Links oficiais:

- API Localidades IBGE - municípios do RS: https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios?orderBy=nome
- Malha Municipal RS 2022: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/RS/RS_Municipios_2022.zip
- Página SES/RS Regionalização: https://saude.rs.gov.br/ageplan-regionalizacao
- XLS Regionalização em Saúde 2025: https://saude.rs.gov.br/upload/arquivos/202507/31162659-regionalizacao-em-saude-2025.xlsx

**Para onde vai?**

É usada para padronizar município em todas as facts e para análises geográficas.


In [41]:
arquivo = PROCESSED_DIR / "dim_municipio_rs.parquet"
df = pd.read_parquet(arquivo)
inspect_dataframe(df)
df.head()

linhas: 497
colunas: 10


,municipio_id_ibge,municipio_nome,municipio_nome_normalizado,uf,regiao_saude,macroregiao_saude,latitude,longitude,populacao,area_km2
0,4300034,Aceguá,acegua,RS,22 - Pampa,Sul,-31.690648,-54.139181,4251,1551.339
1,4300059,Água Santa,agua santa,RS,18 - Araucárias,Norte,-28.213856,-52.051741,4003,291.503
2,4300109,Agudo,agudo,RS,01 - Verdes Campos,Centro-Oeste,-29.608087,-53.226581,16341,534.624
3,4300208,Ajuricaba,ajuricaba,RS,13 - Diversidade,Missioneira,-28.209931,-53.737199,6843,322.674
4,4300307,Alecrim,alecrim,RS,14 - Fronteira Noroeste,Missioneira,-27.651751,-54.783217,6219,316.394


,coluna,dtype,nulos,nulos_pct
0,municipio_id_ibge,object,0,0.0
1,municipio_nome,object,0,0.0
2,municipio_nome_normalizado,object,0,0.0
3,uf,object,0,0.0
4,regiao_saude,object,0,0.0
5,macroregiao_saude,object,0,0.0
6,latitude,float64,0,0.0
7,longitude,float64,0,0.0
8,populacao,int64,0,0.0
9,area_km2,float64,0,0.0


,municipio_id_ibge,municipio_nome,municipio_nome_normalizado,uf,regiao_saude,macroregiao_saude,latitude,longitude,populacao,area_km2
0,4300034,Aceguá,acegua,RS,22 - Pampa,Sul,-31.690648,-54.139181,4251,1551.339
1,4300059,Água Santa,agua santa,RS,18 - Araucárias,Norte,-28.213856,-52.051741,4003,291.503
2,4300109,Agudo,agudo,RS,01 - Verdes Campos,Centro-Oeste,-29.608087,-53.226581,16341,534.624
3,4300208,Ajuricaba,ajuricaba,RS,13 - Diversidade,Missioneira,-28.209931,-53.737199,6843,322.674
4,4300307,Alecrim,alecrim,RS,14 - Fronteira Noroeste,Missioneira,-27.651751,-54.783217,6219,316.394


In [42]:
validacoes_dim = {
    "municipios": len(df),
    "codigos_7_digitos": df["municipio_id_ibge"].astype(str).str.fullmatch(r"43\d{5}").all(),
    "latitude_rs": df["latitude"].between(-34.0, -27.0).all(),
    "longitude_rs": df["longitude"].between(-58.0, -49.0).all(),
}

validacoes_dim

{'municipios': 497,
 'codigos_7_digitos': True,
 'latitude_rs': True,
 'longitude_rs': True}

#### processed/fact_leitos_cnes.parquet

**O que é?**

Oferta de leitos por município, ano, mês e tipo de leito. A métrica principal do MVP é `leitos_sus`.

**De onde vem?**

Vem de `data/interim/cnes/`, processado por `src/data_cleaning/clean_cnes.py`.

Links oficiais de origem:

- FTP CNES LT: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/LT/
- Página CNES Recursos Físicos/DATASUS: https://datasus.saude.gov.br/cnes-recursos-fisicos/
- CNES/DATASUS - tipos de leito: https://cnes2.datasus.gov.br/Mod_Ind_Tipo_Leito.asp?VEstado=00
- CONASS - tabela de domínio CNES leito: https://wiki.conass.org.br/index.php?title=Tabela_de_dom%C3%ADnio_CNES_leito

**Para onde vai?**

Vai para features de oferta, como leitos SUS por 1.000 habitantes e indicador de município sem leito do tipo.


In [43]:
arquivo = PROCESSED_DIR / "fact_leitos_cnes.parquet"
df = pd.read_parquet(arquivo)
inspect_dataframe(df)
df.head()

linhas: 1,007
colunas: 8


,municipio_id,ano,mes,tipo_leito,leitos_existentes,leitos_sus,leitos_nao_sus,estabelecimentos_com_leito
0,4300034,2024,12,leitos_cirurgicos,11,7,4,1
1,4300034,2024,12,leitos_clinicos,17,15,2,1
2,4300034,2024,12,leitos_obstetricos,7,5,2,1
3,4300034,2024,12,leitos_pediatricos,6,6,0,1
4,4300034,2024,12,nao_mapeado,1,1,0,1


,coluna,dtype,nulos,nulos_pct
0,municipio_id,string,0,0.0
1,ano,int64,0,0.0
2,mes,int64,0,0.0
3,tipo_leito,string,0,0.0
4,leitos_existentes,int64,0,0.0
5,leitos_sus,int64,0,0.0
6,leitos_nao_sus,int64,0,0.0
7,estabelecimentos_com_leito,int64,0,0.0


,municipio_id,ano,mes,tipo_leito,leitos_existentes,leitos_sus,leitos_nao_sus,estabelecimentos_com_leito
0,4300034,2024,12,leitos_cirurgicos,11,7,4,1
1,4300034,2024,12,leitos_clinicos,17,15,2,1
2,4300034,2024,12,leitos_obstetricos,7,5,2,1
3,4300034,2024,12,leitos_pediatricos,6,6,0,1
4,4300034,2024,12,nao_mapeado,1,1,0,1


In [44]:
df.groupby("tipo_leito").agg(
    municipios=("municipio_id", "nunique"),
    leitos_sus=("leitos_sus", "sum"),
    leitos_existentes=("leitos_existentes", "sum"),
    estabelecimentos=("estabelecimentos_com_leito", "sum"),
).sort_values("leitos_sus", ascending=False)

,municipios,leitos_sus,leitos_existentes,estabelecimentos
tipo_leito,,,,
leitos_clinicos,235,9956,14066,314
nao_mapeado,192,5435,8227,301
leitos_cirurgicos,190,4043,6731,263
leitos_pediatricos,212,1830,2433,252
leitos_obstetricos,178,1607,2391,215


#### processed/fact_internacoes_sih.parquet

**O que é?**

Internações hospitalares SUS. Esta fact preserva a diferença entre município de residência e município de internação.

**De onde vem?**

Vem de `data/interim/sih/`, processado por `src/data_cleaning/clean_sih.py`.

Links oficiais de origem:

- FTP SIH RD: ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/
- Página Produção Hospitalar SIH/SUS: https://datasus.saude.gov.br/acesso-a-informacao/producao-hospitalar-sih-sus/

**Para onde vai?**

Vai para features de demanda, permanência, mortalidade hospitalar, uso de UTI e saldo regional de pacientes.


In [56]:
arquivo = PROCESSED_DIR / "fact_internacoes_sih.parquet"
df = pd.read_parquet(arquivo)
inspect_dataframe(df)
df.head()
grupo_procedimento = df["grupo_procedimento"].unique()
grupo_procedimento

linhas: 67,069
colunas: 12


,municipio_residencia_id,municipio_internacao_id,ano,mes,grupo_procedimento,diagnostico_cid,idade,sexo,dias_permanencia,valor_aih,obito,uso_uti
0,4300034,4300034,2024,12,procedimentos_clinicos,J159,7,ignorado,3,606.42,False,False
1,4300034,4300034,2024,12,procedimentos_clinicos,E105,65,ignorado,6,408.80,False,False
2,4300034,4300034,2024,12,procedimentos_clinicos,N110,38,ignorado,3,204.50,False,False
3,4300034,4300034,2024,12,procedimentos_clinicos,E440,76,ignorado,5,486.78,False,False
4,4300034,4300034,2024,12,procedimentos_clinicos,A085,91,masculino,2,340.90,False,False


,coluna,dtype,nulos,nulos_pct
0,municipio_residencia_id,string,0,0.0
1,municipio_internacao_id,string,0,0.0
2,ano,Int64,0,0.0
3,mes,Int64,0,0.0
4,grupo_procedimento,string,0,0.0
5,diagnostico_cid,string,0,0.0
6,idade,Int64,0,0.0
7,sexo,string,0,0.0
8,dias_permanencia,int64,0,0.0
9,valor_aih,float64,0,0.0


<StringArray>
[    'procedimentos_clinicos',   'procedimentos_cirurgicos',
               'transplantes', 'procedimentos_diagnosticos']
Length: 4, dtype: string

In [46]:
resumo_sih = {
    "internacoes": len(df),
    "municipios_residencia": df["municipio_residencia_id"].nunique(),
    "municipios_internacao": df["municipio_internacao_id"].nunique(),
    "dias_permanencia": int(df["dias_permanencia"].sum()),
    "valor_aih": float(df["valor_aih"].sum()),
    "obitos_hospitalares": int(df["obito"].sum()),
    "internacoes_com_uti": int(df["uso_uti"].sum()),
}

resumo_sih

{'internacoes': 67069,
 'municipios_residencia': 497,
 'municipios_internacao': 224,
 'dias_permanencia': 394474,
 'valor_aih': 122683399.32999998,
 'obitos_hospitalares': 3413,
 'internacoes_com_uti': 6057}

In [47]:
df.groupby("grupo_procedimento").agg(
    internacoes=("grupo_procedimento", "size"),
    dias_permanencia=("dias_permanencia", "sum"),
    obitos=("obito", "sum"),
    uso_uti=("uso_uti", "sum"),
).sort_values("internacoes", ascending=False)

,internacoes,dias_permanencia,obitos,uso_uti
grupo_procedimento,,,,
procedimentos_clinicos,36966,293118,2942,3498
procedimentos_cirurgicos,29551,96786,465,2477
transplantes,411,3870,4,67
procedimentos_diagnosticos,141,700,2,15


#### processed/fact_populacao_ibge.parquet

**O que é?**

População municipal por ano, com população total e denominadores demográficos específicos para features.

**Estado atual**

A fact foi reconstruída com `497 municípios x 7 anos = 3479` linhas para `2019-2025`.

Campos preenchidos:

- `populacao_total`;
- `pop_0_14`;
- `pop_15_49_feminina`;
- `pop_60_mais`.

As três faixas etárias usam a estrutura municipal do Censo 2022 como proporção aplicada à população total anual. Isso permite calcular denominadores para pediatria, obstetrícia e UTI Adulto já no MVP, mas deve permanecer descrito como aproximação metodológica.

**De onde vem?**

Vem de arquivos do IBGE em `data/raw/ibge/`, processados por `src/data_cleaning/clean_ibge.py`.

Links oficiais de origem:

- SIDRA tabela 6579 - população residente estimada: https://apisidra.ibge.gov.br/values/t/6579/n6/in%20n3%2043/v/9324/p/2019-2025?formato=json
- SIDRA tabela 4714 - Censo 2022/população residente: https://apisidra.ibge.gov.br/values/t/4714/n6/in%20n3%2043/v/93/p/2022?formato=json
- SIDRA tabela 9514 - idade e sexo/Censo 2022: https://apisidra.ibge.gov.br/values/t/9514/n6/in%20n3%2043/v/93/p/2022/c2/4,5/c286/113635/c287/93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93095,93096,93097,93098,49108,49109,60040,60041,6653?formato=json
- IBGE/TCU - população dos municípios para publicação no DOU: https://www.ibge.gov.br/estatisticas/sociais/populacao/37734-relacao-da-populacao-dos-municipios-para-publicacao-no-tcu.html
- XLS IBGE/TCU 2023 municípios: https://ftp.ibge.gov.br/Informacoes_Gerais_e_Referencia/Relacao_da_Populacao_dos_Municipios_para_publicacao_no_DOU_em_2023/POP_TCU_2023_Municipios_POP2022_Malha2023.xls
- XLS IBGE POP2025: https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2025/POP2025_20251031.xls

**Para onde vai?**

Vai para denominadores das features, como internações por 1.000 habitantes, leitos por 1.000 habitantes, leitos pediátricos por crianças, leitos obstétricos por mulheres em idade fértil e UTI Adulto por população adulta/idosa.


In [48]:
arquivo = PROCESSED_DIR / "fact_populacao_ibge.parquet"
df = pd.read_parquet(arquivo)
inspect_dataframe(df)
df.head()

linhas: 497
colunas: 6


,municipio_id,ano,populacao_total,pop_0_14,pop_15_49_feminina,pop_60_mais
0,4300034,2025,4251,<NA>,<NA>,<NA>
1,4300059,2025,4003,<NA>,<NA>,<NA>
2,4300109,2025,16341,<NA>,<NA>,<NA>
3,4300208,2025,6843,<NA>,<NA>,<NA>
4,4300307,2025,6219,<NA>,<NA>,<NA>


,coluna,dtype,nulos,nulos_pct
0,municipio_id,string,0,0.0
1,ano,Int64,0,0.0
2,populacao_total,Int64,0,0.0
3,pop_0_14,Int64,497,100.0
4,pop_15_49_feminina,Int64,497,100.0
5,pop_60_mais,Int64,497,100.0


,municipio_id,ano,populacao_total,pop_0_14,pop_15_49_feminina,pop_60_mais
0,4300034,2025,4251,<NA>,<NA>,<NA>
1,4300059,2025,4003,<NA>,<NA>,<NA>
2,4300109,2025,16341,<NA>,<NA>,<NA>
3,4300208,2025,6843,<NA>,<NA>,<NA>
4,4300307,2025,6219,<NA>,<NA>,<NA>


In [49]:
df.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    populacao_total=("populacao_total", "sum"),
    menor_populacao=("populacao_total", "min"),
    maior_populacao=("populacao_total", "max"),
)

,municipios,populacao_total,menor_populacao,maior_populacao
ano,,,,
2025,497,11233263,1156,1388794


#### processed/fact_nascimentos_sinasc.parquet

**O que é?**

Nascidos vivos e indicadores relacionados ao nascimento por município e ano.

**De onde vem?**

Vem de `data/interim/sinasc/`, processado por `src/data_cleaning/clean_sinasc.py`.

Links oficiais de origem:

- FTP SINASC DNRES: ftp://ftp.datasus.gov.br/dissemin/publicos/SINASC/1996_/Dados/DNRES/
- Página SINASC/DATASUS: https://datasus.saude.gov.br/nascidos-vivos-desde-1994

**Para onde vai?**

Vai para features de leitos obstétricos e, futuramente, UTI neonatal.


In [50]:
arquivo = PROCESSED_DIR / "fact_nascimentos_sinasc.parquet"
df = pd.read_parquet(arquivo)
inspect_dataframe(df)
df.head()

linhas: 497
colunas: 7


,municipio_id,ano,nascidos_vivos,prematuros,baixo_peso,partos_cesareos,partos_vaginais
0,4300034,2024,57,4,3,42,15
1,4300059,2024,39,5,6,29,10
2,4300109,2024,158,19,15,106,52
3,4300208,2024,74,7,5,56,18
4,4300307,2024,38,3,3,32,6


,coluna,dtype,nulos,nulos_pct
0,municipio_id,string,0,0.0
1,ano,Int64,0,0.0
2,nascidos_vivos,int64,0,0.0
3,prematuros,int64,0,0.0
4,baixo_peso,int64,0,0.0
5,partos_cesareos,int64,0,0.0
6,partos_vaginais,int64,0,0.0


,municipio_id,ano,nascidos_vivos,prematuros,baixo_peso,partos_cesareos,partos_vaginais
0,4300034,2024,57,4,3,42,15
1,4300059,2024,39,5,6,29,10
2,4300109,2024,158,19,15,106,52
3,4300208,2024,74,7,5,56,18
4,4300307,2024,38,3,3,32,6


In [51]:
df.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    nascidos_vivos=("nascidos_vivos", "sum"),
    prematuros=("prematuros", "sum"),
    baixo_peso=("baixo_peso", "sum"),
    partos_cesareos=("partos_cesareos", "sum"),
    partos_vaginais=("partos_vaginais", "sum"),
)

,municipios,nascidos_vivos,prematuros,baixo_peso,partos_cesareos,partos_vaginais
ano,,,,,,
2024,497,111988,14552,11122,73590,38352


#### processed/fact_mortalidade_sim.parquet

**O que é?**

Óbitos por município e ano, com grupos relevantes para o projeto.

**De onde vem?**

Vem de `data/interim/sim/`, processado por `src/data_cleaning/clean_sim.py`.

Links oficiais de origem:

- FTP SIM DORES: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/
- Página SIM/DATASUS: https://datasus.saude.gov.br/mortalidade-desde-1996-pela-cid-10

**Para onde vai?**

Vai para features de risco, como mortalidade infantil, neonatal, cardiovascular e respiratória.


In [52]:
arquivo = PROCESSED_DIR / "fact_mortalidade_sim.parquet"
df = pd.read_parquet(arquivo)
inspect_dataframe(df)
df.head()

linhas: 497
colunas: 8


,municipio_id,ano,obitos_total,obitos_neonatais,obitos_infantis,obitos_cardiovasculares,obitos_respiratorios,obitos_queimaduras
0,4300034,2024,36,0,0,14,5,0
1,4300059,2024,28,0,0,8,1,0
2,4300109,2024,179,1,2,64,22,0
3,4300208,2024,73,0,0,22,8,0
4,4300307,2024,99,0,0,43,6,0


,coluna,dtype,nulos,nulos_pct
0,municipio_id,string,0,0.0
1,ano,Int64,0,0.0
2,obitos_total,int64,0,0.0
3,obitos_neonatais,int64,0,0.0
4,obitos_infantis,int64,0,0.0
5,obitos_cardiovasculares,int64,0,0.0
6,obitos_respiratorios,int64,0,0.0
7,obitos_queimaduras,int64,0,0.0


,municipio_id,ano,obitos_total,obitos_neonatais,obitos_infantis,obitos_cardiovasculares,obitos_respiratorios,obitos_queimaduras
0,4300034,2024,36,0,0,14,5,0
1,4300059,2024,28,0,0,8,1,0
2,4300109,2024,179,1,2,64,22,0
3,4300208,2024,73,0,0,22,8,0
4,4300307,2024,99,0,0,43,6,0


In [53]:
df.groupby("ano").agg(
    municipios=("municipio_id", "nunique"),
    obitos_total=("obitos_total", "sum"),
    obitos_neonatais=("obitos_neonatais", "sum"),
    obitos_infantis=("obitos_infantis", "sum"),
    obitos_cardiovasculares=("obitos_cardiovasculares", "sum"),
    obitos_respiratorios=("obitos_respiratorios", "sum"),
)

,municipios,obitos_total,obitos_neonatais,obitos_infantis,obitos_cardiovasculares,obitos_respiratorios
ano,,,,,,
2024,497,101477,811,1137,24961,12312


## Fluxo dos dados no projeto

Resumo do caminho atual dos dados:

1. `data/raw/cnes` e `data/raw/sih`: arquivos `.dbc` históricos baixados do DATASUS para `2019-2025`.
2. `data/raw/ibge`: população total municipal `2019-2025`, com exceções documentadas para `2022` e `2023`.
3. `data/raw/ibge`: estrutura idade/sexo do Censo 2022, usada para preencher denominadores específicos por aproximação proporcional.
4. `data/raw/sinasc` e `data/raw/sim`: arquivos `.dbc` anuais baixados para `2019-2024`.
5. `data/raw/ses`: regionalização usada na dimensão municipal.
6. `data/interim/*`: conversões para Parquet, mantendo granularidade mais próxima da fonte.
7. `data/processed/*`: facts e dimensão municipal; população já foi reconstruída com faixas etárias, enquanto CNES/SIH/SINASC/SIM ainda dependem da conversão histórica para refletirem todo o raw disponível.
8. `src/features/`, `src/models/`, `src/reporting/` e `site/`: próximas etapas de uso analítico.

Ponto metodológico importante: a ausência de leito local não deve ser interpretada sozinha como prioridade automática. Ela é apenas um sinal dentro de um conjunto maior de indicadores.


## Próximas perguntas de análise

Depois deste entendimento inicial, boas perguntas para seguir são:

- A conversão dos 84 arquivos CNES, 84 arquivos SIH, 6 SINASC e 6 SIM para `data/interim/` roda sem perdas?
- As facts hospitalares reconstruídas cobrem o período `2019-2025` na granularidade esperada?
- As facts SINASC/SIM reconstruídas cobrem `2019-2024` com 497 municípios por ano quando aplicável?
- O de-para `TP_LEITO + CODLEITO -> tipo_leito` cobre todos os códigos observados após converter o histórico completo do CNES?
- As internações por residência e por internação contam histórias diferentes para municípios polo?
- As faixas `pop_0_14`, `pop_15_49_feminina` e `pop_60_mais` produzem denominadores coerentes para pediatria, obstetrícia e UTI Adulto?
- Quais features conseguem ser calculadas agora e quais dependem de novas fontes ou novos campos, como SIGTAP e estrutura hospitalar CNES?


## Atualizacao: refino CNES estrutura e SIGTAP

Atualizacao registrada em `2026-06-06` para separar leitos CNES (`LT`) de estrutura hospitalar CNES (`ST/SR/EQ/HB/EF`) e preparar o SIH para refino por SIGTAP.

O objetivo e evitar que uma feature como maternidade, centro cirurgico ou referencia regional seja inferida apenas pela existencia de leito. Leito, estrutura fisica, servico especializado, equipamento e habilitacao sao camadas diferentes da capacidade instalada.


**De onde vem?**

- CNES/DATASUS - diretorio geral dos subconjuntos CNES: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/
- CNES ST - estrutura do estabelecimento: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/ST/
- CNES SR - servicos especializados/classificacoes: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/SR/
- CNES EQ - equipamentos: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/EQ/
- CNES HB - habilitacoes: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/HB/
- CNES EF - estrutura complementar: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/EF/
- SIGTAP/DATASUS - download da tabela unificada: https://sigtap.datasus.gov.br/tabela-unificada/app/download.jsp
- Wiki DATASUS SIGTAP - orientacao de download: https://wiki.datasus.gov.br/sigtap/index.php/Download


**O que mudou na pipeline?**

- Criado `scripts/download_cnes_structure.py` para baixar `ST`, `SR`, `EQ`, `HB` e `EF` no recorte mensal `2019-2025`.
- `src.data_acquisition.convert_dbc` agora aceita `cnes_st`, `cnes_sr`, `cnes_eq`, `cnes_hb` e `cnes_ef`.
- Criado `src.data_cleaning.clean_cnes_structure.build_fact_estrutura_cnes`, gerando `fact_estrutura_cnes.parquet` a partir do `ST`.
- `fact_internacoes_sih` agora preserva `procedimento_codigo`, permitindo join posterior com SIGTAP por procedimento exato.
- Criado `data/external/sigtap_grupos_procedimento.csv` para documentar a classificacao inicial por grupo SIGTAP.


**Campos e features impactadas**

- `CENTRCIR` alimenta `estabelecimentos_com_centro_cirurgico` e `tem_centro_cirurgico`.
- `CENTROBS` e `CENTRNEO` alimentam `tem_maternidade_estrutura` e permitem diferenciar maternidade/obstetricia de simples leito obstetrico.
- `LEITHOSP`, `ATENDHOS` e `VINC_SUS` entram na proxy `tem_referencia_hospitalar_proxy`.
- `procedimento_codigo` no SIH prepara `procedimentos_cirurgicos_12m`, `partos_12m`, `partos_cesareos`, `partos_vaginais` e outras tags finas por SIGTAP.


**Limite metodologico**

`tem_referencia_hospitalar_proxy` e uma proxy tecnica baseada em estrutura registrada no CNES. Ela nao representa, sozinha, a rede formal de referencia pactuada pela regulacao, nem substitui validacao com SES/RS, CIRs, macrorregioes ou fluxos assistenciais observados no SIH.


### Execucao local do CNES ST

Em `2026-06-06`, foi baixado o historico completo `CNES ST` do RS para `2019-2025`: 84 arquivos mensais em `data/raw/cnes_st/`.

Tambem foram convertidos 84 arquivos para `data/interim/cnes_st/` e gerada `data/processed/fact_estrutura_cnes.parquet`. A fact possui 41.742 linhas, 497 municipios, 7 anos e 12 meses por ano.

**De onde vem?**

- CNES ST: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/ST/
- Diretorio geral CNES: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/

**Saidas geradas**

- `data/raw/cnes_st/CNES_ST_RS_2019_01_2026-06-06.dbc` a `data/raw/cnes_st/CNES_ST_RS_2025_12_2026-06-06.dbc`.
- `data/interim/cnes_st/*.parquet`.
- `data/processed/fact_estrutura_cnes.parquet`.


### Execucao local do refino SIGTAP no SIH

Em `2026-06-06`, `fact_internacoes_sih.parquet` foi reconstruida com 67.069 linhas e passou a preservar `procedimento_codigo`, permitindo join posterior com a tabela SIGTAP por procedimento exato.

**De onde vem?**

- SIH RD: ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/
- SIGTAP/DATASUS - download da tabela unificada: https://sigtap.datasus.gov.br/tabela-unificada/app/download.jsp
- Wiki DATASUS SIGTAP - orientacao de download: https://wiki.datasus.gov.br/sigtap/index.php/Download

**Observacao metodologica**

A classificacao atual por grupo usa `data/external/sigtap_grupos_procedimento.csv`. O refinamento fino de parto, cesarea, procedimento cirurgico especifico e linhas clinicas ainda depende da tabela SIGTAP completa por competencia.


### Diagnostico de pendencias para features

Diagnostico registrado em `2026-06-06` apos inspecao das facts processadas. Populacao IBGE e CNES ST ja cobrem `2019-2025`. CNES LT, SIH RD, SINASC e SIM possuem dados brutos historicos baixados, mas as facts processadas atuais ainda refletem apenas o recorte convertido localmente (`2024-12` para CNES/SIH e `2024` para SINASC/SIM).

**Principais pendencias**

- Converter e reconstruir historico completo de CNES LT e SIH RD para `2019-2025`.
- Converter e reconstruir historico completo de SINASC e SIM para `2019-2024`.
- Refinar SIGTAP por procedimento/competencia para partos, cesareas, cirurgias e demandas especificas.
- Preservar campos de dias de UTI no SIH, nao apenas `uso_uti`.
- Definir se distancia a polo/maternidade usara proxy CNES ST ou referencia oficial SES/regulacao.
- Ajustar denominador obstetrico para `mulheres_10_49`; hoje a fact tem `pop_15_49_feminina` como aproximacao inicial.

**De onde vem?**

- CNES LT: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/LT/
- CNES ST: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/ST/
- SIH RD: ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/
- SINASC DNRES: ftp://ftp.datasus.gov.br/dissemin/publicos/SINASC/1996_/Dados/DNRES/
- SIM DORES: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/
- SIGTAP: https://sigtap.datasus.gov.br/tabela-unificada/app/download.jsp
- IBGE SIDRA: https://apisidra.ibge.gov.br/home/ajuda


### Reconstrucao historica das facts

Reconstrucao registrada em `2026-06-06`. Foram convertidos os historicos brutos disponiveis e reconstruidas as facts processadas.

**Intermediarios convertidos**

- CNES LT: 84 Parquets mensais em `data/interim/cnes/`.
- SIH RD: 84 Parquets mensais em `data/interim/sih/`.
- SINASC DN: 6 Parquets anuais em `data/interim/sinasc/`.
- SIM DO: 6 Parquets anuais em `data/interim/sim/`.

**Facts reconstruidas**

- `fact_leitos_cnes.parquet`: 87.947 linhas, `2019-2025`, 12 meses por ano.
- `fact_internacoes_sih.parquet`: 5.423.390 linhas, `2019-2025`, 12 meses por ano.
- `fact_estrutura_cnes.parquet`: 41.742 linhas, `2019-2025`, 12 meses por ano.
- `fact_populacao_ibge.parquet`: 3.479 linhas, `2019-2025`.
- `fact_nascimentos_sinasc.parquet`: 2.982 linhas, `2019-2024`.
- `fact_mortalidade_sim.parquet`: 2.982 linhas, `2019-2024`.

**De onde vem?**

- CNES LT: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/LT/
- CNES ST: ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/ST/
- SIH RD: ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/
- SINASC DNRES: ftp://ftp.datasus.gov.br/dissemin/publicos/SINASC/1996_/Dados/DNRES/
- SIM DORES: ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/
- IBGE SIDRA: https://apisidra.ibge.gov.br/home/ajuda
